# 03 · 记忆（Memory）

真实对话需要上下文。本章学习如何用 `RunnableWithMessageHistory` 给链挂载多轮对话记忆，并按 `session_id` 隔离不同会话。

运行前请确保：
- 已激活 `.venv` 虚拟环境
- 已配置 `.env` 中的 `OPENAI_API_KEY`

> 详细概念讲解见 `docs/03-memory.md`。


## 0. 环境检查


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
if not os.getenv('OPENAI_API_KEY'):
    raise SystemExit('✗ 未找到 OPENAI_API_KEY，请复制 .env.example 为 .env 并填写。')
print('✓ 环境就绪，API Key 已加载')


## 1. 构造带 MessagesPlaceholder 的链

历史消息会被插入到 `MessagesPlaceholder("history")` 所在位置。`input` 是每轮的新问题。


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'))

prompt = ChatPromptTemplate.from_messages([
    ('system', '你是一个简洁的助手。'),
    MessagesPlaceholder('history'),
    ('human', '{input}'),
])
chain = prompt | llm | StrOutputParser()


## 2. 用字典按 session_id 保存历史

`get_session_history` 根据 `session_id` 返回对应的历史对象；同一个 `session_id` 的多轮对话会共享上下文。


In [ ]:
store: dict = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='input',
    history_messages_key='history',
)


## 3. 多轮对话验证

同一个 `session_id` 的后续轮次应能引用前面轮次的信息；不同 `session_id` 之间互不干扰。


In [ ]:
cfg = {'configurable': {'session_id': 'user-1'}}
print('第1轮：', chain_with_history.invoke({'input': '我的名字叫小明。'}, config=cfg))
print('第2轮：', chain_with_history.invoke({'input': '我叫什么名字？'}, config=cfg))  # 应能回答“小明”

cfg2 = {'configurable': {'session_id': 'user-2'}}
print('新会话：', chain_with_history.invoke({'input': '我叫什么名字？'}, config=cfg2))  # 应说不知道


## 4. 下一步

- 检索增强（RAG）：运行 `examples/04_rag.py`，参见 `docs/04-retrieval-and-rag.md`

**常见坑**：忘记传入/读取 `session_id` 会导致所有用户共享同一段历史；历史无限增长会超出上下文窗口，需要截断或摘要。
